<a href="https://colab.research.google.com/github/keduog/EDU-AI-Training/blob/main/Day5/Session3/session3_build_something_real.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Session 3 — Build Something Real

**Day 5 · 13:45 – 15:15**

An assistant that answers from **your own documents**, checks **live facts**, and can
call **your own code**.

## Cell 1 — Setup

In [ ]:
!pip install -q -U google-genai

import os, json, time
from google import genai
from google.genai import types

try:
    from google.colab import userdata
    os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
except Exception:
    if not os.environ.get("GEMINI_API_KEY"):
        import getpass
        os.environ["GEMINI_API_KEY"] = getpass.getpass("Paste your Gemini API key: ")

client = genai.Client()
MODEL = "gemini-3.5-flash"

print("Client ready.")

## Cell 2 — The problem: models make things up

Ask about something the model cannot know, and it will often answer anyway —
confidently, plausibly, and wrongly.

In [ ]:
r = client.models.generate_content(
    model=MODEL,
    contents="What is the leave policy for staff at the EDU AI Lab in 2026?")

print(r.text[:500])
print()
print(">>> Read that carefully. It cannot possibly know. Did it say so? <<<")

## Cell 3 — RAG: give it the facts first

**RAG** = Retrieval Augmented Generation.

Instead of hoping the model knows, you *look the answer up first* and put it in the
prompt. Three steps:

1. Find the relevant text
2. Put it in the prompt
3. Instruct: *"answer using ONLY this text"*

In [ ]:
DOCUMENT = """
EDU AI LAB - TRAINING POLICY 2026

1. Attendance
   All trainees must complete five days of instruction.
   A minimum of four days is required for certification.

2. Cloud resources
   Compute instances must be stopped before leaving the lab.
   Storage accounts may remain running.

3. API keys and credentials
   API keys are personal and must never be shared.
   A leaked key must be deleted and regenerated immediately.

4. Equipment
   Laboratory laptops may not leave the building.
   Personal devices may be used with trainer approval.
"""

doc_config = types.GenerateContentConfig(
    system_instruction=(
        "You answer questions about a policy document. "
        "Use ONLY the information in the DOCUMENT section. "
        "If the answer is not in the document, reply exactly: "
        "'That is not covered in the document.' "
        "Never guess and never use outside knowledge."
    ))


def ask_document(question):
    prompt = f"DOCUMENT:\n{DOCUMENT}\n\nQUESTION: {question}"
    r = client.models.generate_content(
        model=MODEL, contents=prompt, config=doc_config)
    return r.text.strip()


print(ask_document("How many days must I attend to be certified?"))

## Cell 4 — Test it properly, including what it should REFUSE

A trustworthy assistant must be willing to say *"I do not know"*. Test that
deliberately — it is the most important test you will run today.

In [ ]:
questions = [
    "How many days must I attend to be certified?",     # in the document
    "Do I have to stop my compute instance?",           # in the document
    "Can I take a lab laptop home?",                    # in the document
    "What is the salary for instructors?",              # NOT in the document
    "Who won the World Cup in 2022?",                   # NOT in the document
]

for q in questions:
    answer = ask_document(q)
    refused = "not covered" in answer.lower()
    mark = "REFUSED " if refused else "ANSWERED"
    print(f"[{mark}] {q}")
    print(f"           {answer[:130]}")
    print()
    time.sleep(1)

### What you should see

The last two questions should be **refused**. If the model answered them, your system
instruction is not strict enough — tighten it and run again.

> This single behaviour — refusing rather than inventing — is what separates a usable
> assistant from a liability. In a defence context it is not optional.

## Cell 5 — Handling a document too big for one prompt

Real documents are long. A simple, effective approach: split into chunks, keep the
chunks that share the most words with the question.

In [ ]:
def split_into_chunks(text, chunk_size=250):
    """Split text into overlapping chunks of roughly chunk_size characters."""
    paragraphs = [p.strip() for p in text.split("\n\n") if p.strip()]
    chunks, current = [], ""
    for p in paragraphs:
        if len(current) + len(p) < chunk_size:
            current += "\n\n" + p
        else:
            if current:
                chunks.append(current.strip())
            current = p
    if current:
        chunks.append(current.strip())
    return chunks


def find_relevant(question, chunks, top_n=2):
    """Very simple keyword overlap. Real systems use embeddings."""
    q_words = set(question.lower().split())
    scored = []
    for c in chunks:
        overlap = len(q_words & set(c.lower().split()))
        scored.append((overlap, c))
    scored.sort(key=lambda x: -x[0])
    return [c for score, c in scored[:top_n] if score > 0]


chunks = split_into_chunks(DOCUMENT)
print(f"Document split into {len(chunks)} chunks\n")

question = "Can I take a laptop out of the building?"
relevant = find_relevant(question, chunks)

print("Most relevant chunk for that question:")
print(relevant[0] if relevant else "(nothing matched)")

> **Note on honesty:** keyword overlap is crude. Production RAG uses *embeddings* —
> vectors that capture meaning, so "laptop" matches "computer equipment". Gemini offers
> an embeddings API, and that is the natural next step after this course.

## Cell 6 — Ground answers in live web search

The model's training has a cutoff date. Google Search grounding gives it **today's**
information, with sources you can check.

In [ ]:
search_config = types.GenerateContentConfig(
    tools=[types.Tool(google_search=types.GoogleSearch())])

r = client.models.generate_content(
    model=MODEL,
    contents="What is the current population of Ethiopia? Give the year of the estimate.",
    config=search_config)

print(r.text)
print()
print("--- SOURCES ---")

meta = r.candidates[0].grounding_metadata
if meta and meta.grounding_chunks:
    for c in meta.grounding_chunks:
        if c.web:
            print(f"  {c.web.title}")
            print(f"    {c.web.uri}")
else:
    print("  (no sources returned - the model may not have needed to search)")

> ### Always print the sources
> An answer you cannot trace is an answer you cannot defend. Grounding without showing
> the sources gives you confident-sounding text with no accountability — which is worse
> than no answer at all.

## Cell 7 — Function calling: let the model use YOUR code

You describe a function. The model decides **when** it is needed. Your code decides
**what** is actually allowed to run.

In [ ]:
def get_trainee_count(day: str) -> dict:
    """Return how many trainees attended a given training day.

    Args:
        day: The day identifier, one of day1, day2, day3, day4, day5.
    """
    attendance = {"day1": 8, "day2": 8, "day3": 8, "day4": 7, "day5": 8}
    return {"day": day, "count": attendance.get(day.lower(), 0)}


def get_room_temperature(room: str) -> dict:
    """Return the current temperature in a given room.

    Args:
        room: The room name, for example 'lab' or 'office'.
    """
    readings = {"lab": 24, "office": 22, "server": 18}
    return {"room": room, "celsius": readings.get(room.lower(), None)}


# The SDK reads the function signature and docstring automatically
tool_config = types.GenerateContentConfig(
    tools=[get_trainee_count, get_room_temperature])

for q in ["How many trainees came on day 4?",
          "Is the server room cooler than the lab?"]:
    r = client.models.generate_content(model=MODEL, contents=q, config=tool_config)
    print("Q:", q)
    print("A:", r.text.strip())
    print()
    time.sleep(1)

### Why this matters

This is how AI reaches your **real systems** — databases, sensors, ticketing systems,
inventories.

Note the safety boundary: the model can only *request* a call. Your Python code decides
what functions exist and what they are permitted to do. That boundary is your security
layer, and you control it entirely.

## Cell 8 — Put it all together: one assistant

Document knowledge + refusal + web grounding, chosen automatically.

In [ ]:
def smart_assistant(question, use_web=False):
    """Answer from the policy document; optionally allow live web search."""
    if use_web:
        config = types.GenerateContentConfig(
            tools=[types.Tool(google_search=types.GoogleSearch())],
            system_instruction=(
                "You are an assistant for the Ethiopian Defense University. "
                "Answer clearly and cite what you used. "
                "Reply in Amharic if the question is in Amharic."))
        contents = question
    else:
        config = doc_config
        contents = f"DOCUMENT:\n{DOCUMENT}\n\nQUESTION: {question}"

    try:
        r = client.models.generate_content(
            model=MODEL, contents=contents, config=config)
        return r.text.strip()
    except Exception as e:
        return f"ERROR: {str(e)[:150]}"


print("--- policy question (document only) ---")
print(smart_assistant("Must I stop my compute instance?"))
print()
print("--- general question (web allowed) ---")
print(smart_assistant("What is the capital of Ethiopia?", use_web=True))

## Cell 9 — Your turn: build for YOUR work

Replace the document with something real from your own department.

In [ ]:
MY_DOCUMENT = """
Put your own text here - a course outline, a department policy,
a set of procedures, a reading list. Anything factual.
"""

my_config = types.GenerateContentConfig(
    system_instruction=(
        "Answer using ONLY the document below. "
        "If the answer is not there, say 'That is not in the document.' "
        "Reply in the same language as the question."))


def my_assistant(question):
    r = client.models.generate_content(
        model=MODEL,
        contents=f"DOCUMENT:\n{MY_DOCUMENT}\n\nQUESTION: {question}",
        config=my_config)
    return r.text.strip()


# print(my_assistant("your question here"))

---

## Prompt, RAG or fine-tune?

| Problem | Best tool | Cost |
|---|---|---|
| "Answers are too long" | system instruction | minutes, free |
| "It does not know our policy" | **RAG** | hours, cheap |
| "It does not know today's news" | search grounding | minutes |
| "It must query our database" | function calling | hours |
| "It writes in the wrong style/language" | fine-tuning (Day 4) | days |
| "It must run with no internet" | self-hosted (Day 4) | hardware |

**Try them in that order.** Most problems are solved in the first two rows.

Yesterday you learned to fine-tune. Today you learn **when not to** — that judgement is
worth more than either technique.

## Checklist

- [ ] My assistant answers from a document I supplied
- [ ] It **refuses** questions the document does not cover
- [ ] I grounded an answer in Google Search and printed the sources
- [ ] I let the model call a Python function I wrote
- [ ] I can explain RAG in one sentence
- [ ] I replaced the document with something from my own work

**Next:** Session 4 — providers, cost, security and the capstone.